# 03 — Matrix Multiplication

**Ladder:** Naive (1 output / pid) → blocked tiles → **K-tiling** → L2-friendly **grouped** program order → Triton.

**Study tip.** Work the teaching sizes on the boards ($6\times 3@3\times 8$, then $6\times 8@8\times 4$ with $B_M,B_N,B_K=3,2,4$) before large benches. Grouped pid ≠ Group GEMM (folder 08).

Official: [Matrix Multiplication](https://triton-lang.org/main/getting-started/tutorials/03-matrix-multiplication.html)

Handwritten pack: [`docs/visual-guide.pdf`](docs/visual-guide.pdf) · figures: [`docs/images/`](docs/images/)

---

## 0. Definition

Let $A \in \mathbb{R}^{M \times K}$, $B \in \mathbb{R}^{K \times N}$. The product $C = AB \in \mathbb{R}^{M \times N}$ is

$$
C_{mn}
=
\sum_{k=0}^{K-1} A_{mk}\, B_{kn}
=
\langle A_{m,:},\, B_{:,n} \rangle.
$$

**Running examples (from the notes).**

| Stage | Product | Blocks |
|-------|---------|--------|
| Naive / early blocked | $A_{6\times 3}\,B_{3\times 8}\to C_{6\times 8}$ | — / $B_M=3,\,B_N=4$ |
| K-tiled | $A_{6\times 8}\,B_{8\times 4}\to C_{6\times 4}$ | $B_M=3,\,B_N=2,\,B_K=4$ |

**FLOPs.** Exactly $2 M N K$ floating-point operations (one multiply and one add per $(m,n,k)$).

**Peak intensity (roofline sketch).** Moving $A$ and $B$ once and writing $C$ once costs

$$
\text{bytes} \approx (MK + KN + MN)\,b,
\qquad
\text{AI} \approx \frac{2 M N K}{(MK + KN + MN)\,b}\ \text{FLOP/byte}.
$$

Reuse (blocking + cache-friendly order) raises effective arithmetic intensity by reducing DRAM reloads of $A$ and $B$.

### Hardware mental model (before kernels)

![Grid → Thread Block → Warp → Threads](docs/images/grid-block-warp-thread.jpeg)

![CUDA hierarchy: grid, blocks, warps of 32, SIMT](docs/images/cuda-execution-hierarchy.png)

Triton **programs** correspond to CUDA blocks. A matmul launch is a 1-D grid; each program owns either one $C_{mn}$ (naive) or one $B_M\times B_N$ tile (blocked / grouped).

### Pitfalls

- Wrong stride order when building a_ptrs/b_ptrs.
- Accumulating FP16 without FP32 acc → large error.
- Confusing tutorial-03 “GROUP_SIZE_M” (L2 swizzle) with tutorial-08 Group GEMM.

### Self-check

1. Hand-map pid→(pm,pn) for BM=3,BN=4,M=6,N=8.
2. Write two K-panel steps for BK=4, K=8.
3. Why does grouped order reduce B-panel reloads?


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pytest
import torch
import triton
import triton.language as tl

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")



## 1. Naive indexing — one $(m,n)$ per program

![Naive: one program per C[m,n]; m=i//N, n=i%N](docs/images/matmul-naive-threads.png)

![Notes — naive overview](docs/images/notes-01-naive-overview.png)

Enumerate output entries by a flat program id $\mathrm{pid} \in \{0,\ldots,MN-1\}$:

$$
m = \left\lfloor \frac{\mathrm{pid}}{N} \right\rfloor,
\qquad
n = \mathrm{pid} \bmod N.
$$

**Lemma (bijection).** The map $\mathrm{pid} \mapsto (m,n)$ is a bijection onto $\{0,\ldots,M-1\} \times \{0,\ldots,N-1\}$.

*Proof.* Quotient–remainder theorem for division by $N$: unique $m,n$ with $\mathrm{pid} = m N + n$ and $0 \le n < N$.

Then

$$
C_{mn} = \sum_{k=0}^{K-1} A_{mk} B_{kn}.
$$

**Concrete check.** For $M=6$, $N=8$ there are $48$ programs; pid $19$ writes $C[2,3]$ because $19//8=2$ and $19\% 8=3$.

$A[m,:]$ is a **contiguous** row load; $B[:,n]$ is a **strided** column load. Correct, but each program does a length-$K$ reduction with poor reuse of $B$'s columns across neighboring $m$.


In [ ]:
def naive_matmul_psuedo(A, B):
    """
    Matmul is simple! We have two matricies:
    (See markdown cells above for the math.)
    """

    M, K = A.shape
    K, N = B.shape
    C = torch.zeros(M, N, device=A.device, dtype=A.dtype)

    def dot(v1, v2):
        accum = 0
        for p, q in zip(v1, v2):
            accum += p * q
        return accum

    num_pids = list(range(M * N))  # [0, 1, 2, ... m*n]
    for i in num_pids:
        ### Get our m,n index from our position in the threads ###
        ### Each row of our output matrix is N long, so we can get ###
        ### which row we are on with a simple division operation ###
        m = i // N

        ### And we can get which column we are on with a simple modulo operation ###
        n = i % N

        C[m, n] = dot(A[m, :], B[:, n])

    return C


## 2. Blocking — tiles of $C$

![Blocked tile $B_M=3$, $B_N=2$](docs/images/notes-05-blocked-tile.png)

![K-tiling: $A_{3\times 4}\,B_{4\times 2}\to C_{3\times 2}$](docs/images/notes-08-k-tiling.png)

![Two $B_K$ panels accumulate into one C-tile](docs/images/notes-13-k-accumulation.png)

Choose tile sizes $B_M, B_N$ (and later $B_K$ along the reduction). Assume for the teaching case $B_M \mid M$ and $B_N \mid N$. Number of tile-rows / tile-cols:

$$
G_M = \frac{M}{B_M},\qquad G_N = \frac{N}{B_N}.
$$

Program $\mathrm{pid} \in \{0,\ldots,G_M G_N - 1\}$ owns output block

$$
(p_m, p_n)
=
\Big(\big\lfloor \mathrm{pid}/G_N \big\rfloor,\;\mathrm{pid}\bmod G_N\Big),
$$

and the corresponding submatrix

$$
C\big[p_m B_M : (p_m+1)B_M,\; p_n B_N : (p_n+1)B_N\big]
=
A\big[p_m B_M:(p_m+1)B_M,:\big]\,
B\big[:, p_n B_N:(p_n+1)B_N\big].
$$

**Teaching walk** ($M=6$, $N=8$, $K=3$, $B_M=3$, $B_N=4$ → $G_M=G_N=2$, four programs):

| pid | $(p_m,p_n)$ | Writes |
|-----|-------------|--------|
| 0 | $(0,0)$ | $C[0:3,0:4]=A[0:3,:]\,B[:,0:4]$ |
| 1 | $(0,1)$ | $C[0:3,4:8]$ |
| 2 | $(1,0)$ | $C[3:6,0:4]$ |
| 3 | $(1,1)$ | $C[3:6,4:8]$ |

**Along $K$.** When $K$ is large (notes: $M,N\sim 10^6$), do **not** take the full strip. Split the reduction into panels of width $B_K$:

$$
C_{\mathrm{tile}}
=
\sum_{t=0}^{\lceil K/B_K \rceil - 1}
A_{\mathrm{tile},t}\, B_{t,\mathrm{tile}},
$$

accumulating in $\mathrm{fp32}$ even if $A,B$ are $\mathrm{fp16}$.

**K-tiled example.** $A_{6\times 8}\,B_{8\times 4}$, $B_M=3$, $B_N=2$, $B_K=4$ → grid $2\times 2=4$ programs; each tile needs **two** `tl.dot` steps ($k$-panels $[0:4)$ then $[4:8)$).

**Row-major tile order** assigns consecutive pids across a row of tiles — good reuse of $A$-rows within a row of $C$-tiles, weaker reuse of $B$-columns when moving to the next row of tiles (L2 may have evicted early $B$ panels).


In [ ]:
def blocked_matmul_pseudo(A, B, BLOCK_SIZE_M, BLOCK_SIZE_N):
    """
    We typically do GPU compute in blocks. Instead of having every
    (See markdown cells above for the math.)
    """

    M, K = A.shape
    K, N = B.shape
    C = torch.zeros(M, N, device=A.device, dtype=A.dtype)

    assert M % BLOCK_SIZE_M == 0, "Assuming M divisible by BLOCK_SIZE_M for simplicity"
    assert N % BLOCK_SIZE_N == 0, "Assuming N divisible by BLOCK_SIZE_N for simplicity"

    ### Get how many rows and cols we have at the Block level ###
    num_block_rows = M // BLOCK_SIZE_M
    num_block_cols = N // BLOCK_SIZE_N

    ### Now each pid refers to a block of our output, not just a single output
    num_pids = list(range(num_block_rows * num_block_cols))
    for i in num_pids:
        ### This gives us the top left index of every block
        block_m = i // num_block_cols  # What row are we on?
        block_n = i % num_block_cols  # What col are we on?

        ### But we need to convert block_m, block_n to actual indexes in our data
        m = block_m * BLOCK_SIZE_M
        n = block_n * BLOCK_SIZE_N

        ### Grab the corresponding block of rows and cols from A and B
        block_A = A[m : m + BLOCK_SIZE_M, :]
        block_B = B[:, n : n + BLOCK_SIZE_N]

        ### Now its basically a tiny matmul between block_A and block_B!
        C[m : m + BLOCK_SIZE_M, n : n + BLOCK_SIZE_N] = naive_matmul_psuedo(block_A, block_B)

    return C


## 3. Grouped ordering — L2-friendly swizzle

![Row-major vs grouped A/B load counts](docs/images/notes-19-group-load-cost.png)

![Grouped pid: column-major inside each group](docs/images/notes-20-grouped-pid.png)

Fix a group height $G$ (Triton: `GROUP_SIZE_M`). Let

$$
\mathrm{num\_pid\_in\_group} = G \cdot G_N.
$$

For program $\mathrm{pid}$:

$$
\begin{aligned}
\mathrm{group\_id} &= \big\lfloor \mathrm{pid} / (G\, G_N) \big\rfloor, \\
\mathrm{first\_pid\_m} &= \mathrm{group\_id}\cdot G, \\
g_m &= \min(G_M - \mathrm{first\_pid\_m},\, G), \\
w &= \mathrm{pid} \bmod (G\, G_N), \\
p_m &= \mathrm{first\_pid\_m} + (w \bmod g_m), \\
p_n &= \big\lfloor w / g_m \big\rfloor.
\end{aligned}
$$

**Interpretation.** Within a group, pids advance in **column-major** order over a $g_m \times G_N$ block of tiles:

$$
\begin{bmatrix}
0 & 2 & \cdots \\
1 & 3 & \cdots
\end{bmatrix}
\quad\text{(example $G=2$)}
\quad\text{instead of row-major}
\quad
\begin{bmatrix}
0 & 1 & \cdots \\
G_N & G_N+1 & \cdots
\end{bmatrix}.
$$

**Notes example.** $A_{6\times 3}\,B_{3\times 8}$, $B_M=3$, $B_N=4$, $G=2$ → $G_M=G_N=2$, `num_pid_in_group=4`. Pid order on the tile grid:

$$
\begin{bmatrix}0&2\\1&3\end{bmatrix}
\quad\Rightarrow\quad
\begin{aligned}
\mathrm{pid}\,0&\to C[0:3,0:4],\\
\mathrm{pid}\,1&\to C[3:6,0:4]\quad\text{(same $B$ column panel)},\\
\mathrm{pid}\,2&\to C[0:3,4:8],\\
\mathrm{pid}\,3&\to C[3:6,4:8].
\end{aligned}
$$

**Load-count sketch from the notes** (writing 9 $C$-tiles, group size $3$):

| Order | A-panel loads | B-panel loads |
|-------|---------------|---------------|
| Row-major | $9$ | $81$ |
| Grouped | $27$ | $27$ |

**Why this helps (informal cache argument).**

1. **Vertical reuse of $B$.** Consecutive pids in a group often share the same $B$-column panel while varying $A$-row panels — column loads of $B$ are expensive in row-major layout, so amortize them.
2. **Locality in both axes.** Nearby pids overlap in $A$-rows **or** $B$-cols, raising L2 hit probability vs pure row-major sweeps that stream all of $B$ before returning.
3. **Partial last group.** $g_m = \min(\cdot)$ handles $G \nmid G_M$ without changing the algebra of $C = AB$.

**Theorem (same $C$).** Grouped ordering only permutes which program computes which tile; each tile formula is identical to §2. Hence $C$ is independent of the swizzle.

*Proof.* The map $\mathrm{pid} \mapsto (p_m,p_n)$ is a bijection onto the tile grid (standard grouped-launch construction); each tile write is disjoint and equals the corresponding block product.


In [ ]:
def grouped_blocked_matmul_psuedo(A, B, BLOCK_SIZE_M, BLOCK_SIZE_N, GROUP_SIZE):
    """
    Blocking our matmul is a step in the right direction as it now enables proper
    (See markdown cells above for the math.)
    """

    M, K = A.shape
    K, N = B.shape
    C = torch.zeros(M, N, device=A.device, dtype=A.dtype)

    assert M % BLOCK_SIZE_M == 0, "Assuming M divisible by BLOCK_SIZE_M for simplicity"
    assert N % BLOCK_SIZE_N == 0, "Assuming N divisible by BLOCK_SIZE_N for simplicity"
    assert (BLOCK_SIZE_M % GROUP_SIZE == 0) and (
        BLOCK_SIZE_N % GROUP_SIZE == 0
    ), "Assuming BLOCK_SIZE_M/N are divisible by our GROUP_SIZE"

    ### Get how many rows and cols we have at the Block level ###
    num_block_rows = M // BLOCK_SIZE_M
    num_block_cols = N // BLOCK_SIZE_N

    num_pids = list(range(num_block_rows * num_block_cols))

    ### To visualize our groups we can just store the PID index in a matrix to print
    pid_visual = torch.zeros(num_block_rows, num_block_cols)
    for i in num_pids:
        ### Get number of Programs in Group ###
        ### This is for all columns, as in our example ###
        ### col_id:    0  1     2  3
        ### row_id: 0 [0  2  |  4  6 ]
        ### row_id: 1 [1  3  |  5  7 ]
        ###           ----------------
        ### row_id: 2 [8  10 | 12 14 ]
        ### row_id: 3 [9  11 | 13 15 ]

        ### to get to 8 (the start of the next group down the column)
        ### We need to get through 2 * 4 blocks. (group_size * num_block_cols)
        num_pid_in_group = GROUP_SIZE * num_block_cols

        ### Get Which Group We Are In (down the column) ###
        group_id = i // num_pid_in_group

        ### Now we can get the Row ID of the first program in the group
        ### This is the top of this specific group, we need to index further
        ### soon to get the exact row we are processing inside the group
        first_pid_m = group_id * GROUP_SIZE

        ### As a sanity check our group_size may not perfectly divide our num blocks
        ### and the last group may be smaller. In this psuedocode we dont worry about
        ### this as we assume divisible, but the triton code will have this so i put it here!
        group_size_m = min(num_block_rows - first_pid_m, GROUP_SIZE)

        ### Now that we have our row we are working on we need the column. And as we saw
        ### we do column major order within groups. (this part is trickier)
        where_in_group = i % num_pid_in_group

        ### Now we need to get ourselves into the correct row, as its column major.
        which_row = where_in_group % group_size_m

        ### And this is the row within the group, we just have to shift it over by where ever our starting group is
        pid_m = first_pid_m + which_row

        ### Now we get the column that this pid is supposed to compute.
        where_in_group = i % num_pid_in_group
        pid_n = where_in_group // group_size_m
        pid_visual[pid_m, pid_n] = i

        ### But we need to convert block_m, block_n to actual indexes in our data
        m = pid_m * BLOCK_SIZE_M
        n = pid_n * BLOCK_SIZE_N

        ### Grab the corresponding block of rows and cols from A and B
        block_A = A[m : m + BLOCK_SIZE_M, :]
        block_B = B[:, n : n + BLOCK_SIZE_N]

        ### Same matmul op on our block of data ###
        C[m : m + BLOCK_SIZE_M, n : n + BLOCK_SIZE_N] = naive_matmul_psuedo(block_A, block_B)

    return C, pid_visual


## 4. Triton kernels — same algebra

| Kernel | Program owns | Indexing |
|---|---|---|
| `naive_matmul_kernel` | one $C_{mn}$ | $m=\lfloor\mathrm{pid}/N\rfloor$, $n=\mathrm{pid}\bmod N$ |
| `blocked_matmul_kernel` | one $B_M \times B_N$ tile | row-major over tiles + $K$-panels of width $B_K$ |
| `grouped_matmul_kernel` | one tile | grouped swizzle of §3 + **same** $K$-loop / `tl.dot` / store |

**Accumulation.** Prefer $\mathrm{fp32}$ accumulator $Acc$, then cast:

$$
C_{\mathrm{tile}} = \mathrm{cast}\Big(\sum_t A_t B_t\Big).
$$

Masked loads implement the indicator $\mathbf{1}[\mathrm{index\ in\ bounds}]$ so shapes need not divide block sizes.

**Pointer form (blocked / grouped body).** With offsets $o_m,o_n,o_k$:

$$
\begin{aligned}
A_{\mathrm{ptrs}} &= a + o_m[:,\texttt{None}]\,s_{am} + o_k[\texttt{None},:]\,s_{ak},\\
B_{\mathrm{ptrs}} &= b + o_k[:,\texttt{None}]\,s_{bk} + o_n[\texttt{None},:]\,s_{bn},
\end{aligned}
$$

then each $K$-step advances $A_{\mathrm{ptrs}}$ by $B_K s_{ak}$ and $B_{\mathrm{ptrs}}$ by $B_K s_{bk}$.

Only the $\mathrm{pid}\mapsto(p_m,p_n)$ block differs between `blocked_matmul_kernel` and `grouped_matmul_kernel`.


In [ ]:
@triton.jit
def naive_matmul_kernel(
    A_ptr,
    B_ptr,
    C_ptr,
    M,
    N,
    K,
    stride_am,
    stride_ak,
    stride_bk,
    stride_bn,
    stride_cm,
    stride_cn,
):
    # Program ID — each "program" computes one element of C
    pid = tl.program_id(axis=0)  # 0..M*N - 1

    # Compute (m, n) coordinates of this output element
    m = pid // N
    n = pid % N

    # If we're out of bounds, exit
    if (m >= M) or (n >= N):
        return

    # Compute the dot product between A[m, :] and B[:, n]
    acc = 0.0
    for k in range(0, K):
        a_val = tl.load(A_ptr + m * stride_am + k * stride_ak)
        b_val = tl.load(B_ptr + k * stride_bk + n * stride_bn)
        acc += a_val * b_val

    # Write result to output
    tl.store(C_ptr + m * stride_cm + n * stride_cn, acc)


def naive_matmul(A, B):
    M, K = A.shape
    K2, N = B.shape
    assert K == K2

    C = torch.zeros((M, N), device=A.device, dtype=A.dtype)

    grid = (M * N,)

    naive_matmul_kernel[grid](
        A,
        B,
        C,
        M,
        N,
        K,
        A.stride(0),
        A.stride(1),
        B.stride(0),
        B.stride(1),
        C.stride(0),
        C.stride(1),
    )
    return C


In [ ]:
def get_blocked_cuda_autotune_config():
    return [
        triton.Config({"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 256, "BLOCK_SIZE_K": 64}, num_stages=3, num_warps=8),
        triton.Config({"BLOCK_SIZE_M": 64, "BLOCK_SIZE_N": 256, "BLOCK_SIZE_K": 32}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 128, "BLOCK_SIZE_K": 32}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 64, "BLOCK_SIZE_K": 32}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_SIZE_M": 64, "BLOCK_SIZE_N": 128, "BLOCK_SIZE_K": 32}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 32, "BLOCK_SIZE_K": 32}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_SIZE_M": 64, "BLOCK_SIZE_N": 32, "BLOCK_SIZE_K": 32}, num_stages=5, num_warps=2),
        triton.Config({"BLOCK_SIZE_M": 32, "BLOCK_SIZE_N": 64, "BLOCK_SIZE_K": 32}, num_stages=5, num_warps=2),
    ]


@triton.autotune(configs=get_blocked_cuda_autotune_config(), key=["M", "N", "K"])
@triton.jit
def blocked_matmul_kernel(
    A_ptr,
    B_ptr,
    C_ptr,
    M,
    N,
    K,
    stride_am,
    stride_ak,
    stride_bk,
    stride_bn,
    stride_cm,
    stride_cn,
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
):
    pid = tl.program_id(axis=0)

    ### Get how many columns of blocks we have ###
    num_block_cols = tl.cdiv(N, BLOCK_SIZE_N)

    ### Recover our actual block index ###
    block_m = pid // num_block_cols  # What row are we on?
    block_n = pid % num_block_cols  # What col are we on?

    ### Compute Offsets to grab full block of data ###
    offs_m = block_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_n = block_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    offs_k = tl.arange(0, BLOCK_SIZE_K)

    a_ptrs = A_ptr + (offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = B_ptr + (offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn)

    ### Initializer Accumulator - use float32 for accumulation ###
    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)

    ### Loop over K in chunks of BLOCK_SIZE_K ###
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        ### Load Blocks of A and B with proper masking ###
        # Mask for A: check both M and K dimensions
        a_mask = (offs_m[:, None] < M) & (offs_k[None, :] < K - k * BLOCK_SIZE_K)
        a = tl.load(a_ptrs, mask=a_mask, other=0.0)

        # Mask for B: check both K and N dimensions
        b_mask = (offs_k[:, None] < K - k * BLOCK_SIZE_K) & (offs_n[None, :] < N)
        b = tl.load(b_ptrs, mask=b_mask, other=0.0)

        ### Accumulate Result ###
        ### investigate later: There is a discrepancy between pytorch and triton in float32
        ### if we dont include allow_fp32=False we wil have a higher error, but also,
        ### we then hurt performance on float32. Leaving as is for now!
        ###
        ### This is a well documented issue
        ### https://github.com/triton-lang/triton/issues/4574
        ### https://github.com/triton-lang/triton/issues/5204
        ### https://github.com/triton-lang/triton/issues/2843

        accumulator += tl.dot(a, b)  # allow_tf32=False

        ### Advance our Offsets to the next chunk of K ###
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk

    ### Cast accumulator to correct dtype ###
    c = accumulator.to(A_ptr.dtype.element_ty)

    ### Identify which block in our output C we will save this in ###
    c_offsets = stride_cm * offs_m[:, None] + stride_cn * offs_n[None, :]

    ### Identify any invalid positions we dont want to save in ###
    c_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)

    ### Save it! ###
    tl.store(C_ptr + c_offsets, c, mask=c_mask)


def blocked_matmul(a, b):
    # Get dimensions
    M, K = a.shape
    K2, N = b.shape
    assert K == K2, "Inner dimensions must match"

    # Initialize output tensor with same dtype as input
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)

    # Define grid
    grid = lambda meta: (triton.cdiv(M, meta["BLOCK_SIZE_M"]) * triton.cdiv(N, meta["BLOCK_SIZE_N"]),)

    # Launch kernel
    blocked_matmul_kernel[grid](
        a,
        b,
        c,
        M,
        N,
        K,
        a.stride(0),
        a.stride(1),
        b.stride(0),
        b.stride(1),
        c.stride(0),
        c.stride(1),
    )
    return c


In [ ]:
def get_cuda_autotune_config():
    return [
        triton.Config(
            {"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 256, "BLOCK_SIZE_K": 64, "GROUP_SIZE_M": 8},
            num_stages=3,
            num_warps=8,
        ),
        triton.Config(
            {"BLOCK_SIZE_M": 64, "BLOCK_SIZE_N": 256, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8},
            num_stages=4,
            num_warps=4,
        ),
        triton.Config(
            {"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 128, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8},
            num_stages=4,
            num_warps=4,
        ),
        triton.Config(
            {"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 64, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8},
            num_stages=4,
            num_warps=4,
        ),
        triton.Config(
            {"BLOCK_SIZE_M": 64, "BLOCK_SIZE_N": 128, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8},
            num_stages=4,
            num_warps=4,
        ),
        triton.Config(
            {"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 32, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8},
            num_stages=4,
            num_warps=4,
        ),
        triton.Config(
            {"BLOCK_SIZE_M": 64, "BLOCK_SIZE_N": 32, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8},
            num_stages=5,
            num_warps=2,
        ),
        triton.Config(
            {"BLOCK_SIZE_M": 32, "BLOCK_SIZE_N": 64, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8},
            num_stages=5,
            num_warps=2,
        ),
    ]


@triton.autotune(configs=get_cuda_autotune_config(), key=["M", "N", "K"])
@triton.jit
def grouped_matmul_kernel(
    A_ptr,
    B_ptr,
    C_ptr,
    M,
    N,
    K,
    stride_am,
    stride_ak,
    stride_bk,
    stride_bn,
    stride_cm,
    stride_cn,
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
    GROUP_SIZE_M: tl.constexpr,
    DTYPE_FLAG: tl.constexpr,  # 0 for float32, 1 for float16
):
    ### Grouping logic as described above! ###
    pid = tl.program_id(axis=0)

    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    ### EVERYTHING ELSE IS THE SAME AS BEFORE! ###
    ### Compute Offsets to grab full block of data ###
    offs_m = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_n = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    offs_k = tl.arange(0, BLOCK_SIZE_K)

    a_ptrs = A_ptr + (offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = B_ptr + (offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn)

    ### Initializer Accumulator - use float32 for accumulation ###
    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)

    ### Loop over K in chunks of BLOCK_SIZE_K ###
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        ### Load Blocks of A and B with proper masking ###
        # Mask for A: check both M and K dimensions
        a_mask = (offs_m[:, None] < M) & (offs_k[None, :] < K - k * BLOCK_SIZE_K)
        a = tl.load(a_ptrs, mask=a_mask, other=0.0)

        # Mask for B: check both K and N dimensions
        b_mask = (offs_k[:, None] < K - k * BLOCK_SIZE_K) & (offs_n[None, :] < N)
        b = tl.load(b_ptrs, mask=b_mask, other=0.0)

        ### Accumulate Result ###
        accumulator += tl.dot(a, b)

        ### Advance our Offsets to the next chunk of K ###
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk

    ### Cast accumulator to correct dtype ###
    c = accumulator.to(A_ptr.dtype.element_ty)

    ### Identify which block in our output C we will save this in ###
    c_offsets = stride_cm * offs_m[:, None] + stride_cn * offs_n[None, :]

    ### Identify any invalid positions we dont want to save in ###
    c_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)

    ### Save it! ###
    tl.store(C_ptr + c_offsets, c, mask=c_mask)


def fused_grouped_matmul(a, b):
    # Get dimensions
    M, K = a.shape
    K2, N = b.shape
    assert K == K2, "Inner dimensions must match"
    assert (a.dtype == torch.float32) or (a.dtype == torch.float16), "Only support float32 or float16 ops"
    assert a.dtype == b.dtype, f"a and b have different precision types {a.dtype} and {b.dtype}"

    # Initialize output tensor with same dtype as input
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)

    # Define grid
    grid = lambda meta: (triton.cdiv(M, meta["BLOCK_SIZE_M"]) * triton.cdiv(N, meta["BLOCK_SIZE_N"]),)

    # Launch kernel
    grouped_matmul_kernel[grid](
        a,
        b,
        c,
        M,
        N,
        K,
        a.stride(0),
        a.stride(1),
        b.stride(0),
        b.stride(1),
        c.stride(0),
        c.stride(1),
        DTYPE_FLAG=0 if a.dtype == torch.float32 else 1,
    )

    return c


# Alias used by later tutorials (08/09/10)
def matmul(a, b, activation=""):
    del activation  # kept for API compatibility with earlier tutorial versions
    return fused_grouped_matmul(a, b)


In [ ]:
@pytest.mark.parametrize(
    "M, N, K",
    [
        (64, 64, 64),
        (128, 128, 128),
        (256, 256, 256),
        (512, 512, 512),
        (256, 512, 128),
        (512, 256, 128),
    ],
)
def test_matmul(M, N, K):
    require_gpu()
    # Create random inputs
    a = torch.randn(size=(M, K), dtype=torch.float16, device=DEVICE)
    b = torch.randn(size=(K, N), dtype=torch.float16, device=DEVICE)

    out_ref = torch.matmul(a, b)

    # Your fused kernel
    out = fused_grouped_matmul(a, b)

    # Validate results
    assert torch.allclose(out, out_ref, atol=1e-2, rtol=1e-2)

    # Log useful info
    max_diff = float(torch.max(torch.abs(out - out_ref)))
    print(f"Matmul Success: M={M}, N={N}, K={K} | Max Diff={max_diff:.4f}")


def test_correctness() -> None:
    test_matmul(64, 64, 64)


def test_pseudo_matches_torch() -> None:
    torch.manual_seed(0)
    a = torch.randn(6, 3)
    b = torch.randn(3, 8)
    ref = a @ b
    assert torch.allclose(naive_matmul_psuedo(a, b), ref, atol=1e-5)
    assert torch.allclose(blocked_matmul_pseudo(a, b, 3, 4), ref, atol=1e-5)
    grouped, _ = grouped_blocked_matmul_psuedo(a, b, 3, 4, GROUP_SIZE=1)
    assert torch.allclose(grouped, ref, atol=1e-5)


In [ ]:
configs = [
    triton.testing.Benchmark(
        x_names=["M", "N", "K"],
        x_vals=[512 * i for i in range(2, 9)],
        line_arg="provider",
        line_vals=["torch", "blocked_matmul", "grouped_matmul"],
        line_names=["PyTorch", "Matmul No Groups", "Matmul Grouped"],
        styles=[("blue", "-"), ("purple", "-"), ("green", "-")],
        ylabel="TFLOPS",
        plot_name="matmul",
        args={},
    )
]


@triton.testing.perf_report(configs)
def benchmark_matmul(M, N, K, provider):
    if provider == "torch":
        a = torch.randn((M, K), device=DEVICE, dtype=torch.float16)
        b = torch.randn((K, N), device=DEVICE, dtype=torch.float16)
        ms = triton.testing.do_bench(lambda: torch.matmul(a, b))
    if provider == "blocked_matmul":
        a = torch.randn((M, K), device=DEVICE, dtype=torch.float16)
        b = torch.randn((K, N), device=DEVICE, dtype=torch.float16)
        ms = triton.testing.do_bench(lambda: blocked_matmul(a, b))
    if provider == "grouped_matmul":
        a = torch.randn((M, K), device=DEVICE, dtype=torch.float16)
        b = torch.randn((K, N), device=DEVICE, dtype=torch.float16)
        ms = triton.testing.do_bench(lambda: fused_grouped_matmul(a, b))

    perf = lambda ms: 2 * M * N * K * 1e-12 / (ms * 1e-3)
    return perf(ms)
